# LLM Gateway with Portkey: Zero to production

In [1]:
import os
import time
import uuid
from dotenv import load_dotenv
import json
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

In [2]:
load_dotenv(dotenv_path="../.env")

# Portkey api key - authenticates your app to the Portkey gateway
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY","3mlMZJ+1VcLRdHJHmGtGj4iDVzGm")


GROQ_SLUG = "rag"
GROQ_MODEL = f"@{GROQ_SLUG}/llama-3.3-70b-versatile"


GROQ_SLUG_2 = 'brag'
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"

# Keep GROQ_API_KEY for the LangChain experiment (Exp -9)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
# helper functions

def section(title):
    print(f"\n{'='}*62")
    print(f"{title}")
    print(f"\n{'='}*62")


def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱️ {ms:.0f}ms{note}")
    print(bar)


# main portkey client - used for all simple experiments
portkey = Portkey(api_key=PORTKEY_API_KEY)

print("Setup Colplete!")
print(f" Portkey API Key : {'Ok' if PORTKEY_API_KEY else "Missing"}")
print(f"\n Portkey Gateway : {PORTKEY_GATEWAY_URL}")

Setup Colplete!
 Portkey API Key : Ok

 Portkey Gateway : https://api.portkey.ai/v1


# BASELINE - DIRECT LLM Call (NO Gateway)

Goal: See what a raw LLM call look like - no routing, no loggin, no resilience.

In [4]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage



raw_groq = ChatGroq(api_key=GROQ_API_KEY,model="llama-3.3-70b-versatile", temperature=0)

section("BASELINE - Direct Groq Call")

question = [
    "what is kubernetes in one sentence?",
    "What is Intel SRIOV?"
]

for q in question:
    t0 = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time() - t0)*1000, label='direct Groq - no gateqay')

print("\n Missing: logging, retries, fallback, caching, metadata.")

d:\Project\RAG PROJECT\enterprise_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=*62
BASELINE - Direct Groq Call

=*62

──────────────────────────────────────────────────────────────
Q: what is kubernetes in one sentence?
A: Kubernetes is an open-source container orchestration system that automates the deployment, scaling, and management of containerized applications across a cluster of machines.
⏱️ 1789ms | direct Groq - no gateqay
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: Intel SR-IOV (Single Root I/O Virtualization) is a technology that allows a single physical device, such as a network interface card (NIC) or a storage controller, to appear as multiple virtual devices to the operating system and applications. This is achieved...
⏱️ 2038ms | direct Groq - no gateqay
──────────────────────────────────────────────────────────────

 Missing: logging, retries, fallback, caching, metadata.


# Experiment - 1 Route Through the GateWay

In [5]:
section("EXP 1 - Basic Gateway call")


# for q in question:
#     t0 = time.time()
#     r = portkey.chat.completions.create(
#         model=GROQ_MODEL,
#         messages=[{'role':'user', 'content':q}]
#     )
#     show(q, r.choices[0].message.content, (time.time()-t0)*1000,
#          label="routed via Portkey gateway")


=*62
EXP 1 - Basic Gateway call

=*62


# EXPERIMENT 2 - Metadata & Observability

In [6]:
section("EXP 2 - Metadata & Observability")

session = str(uuid.uuid4())[:8]

scenarios = [
    ('alice','enterprise-rag', "What is Kubernetes RBAC?"),
    ('bob', "docs-chatbot", "How does BGP path selection work?"),
    ('carol', "support-bot", "What is SRIOV virtualization?"),
    ('alice', "enterprise-rag", "Explain Kubernetes NetworkPilice")
]


# for user, feature, q in scenarios:
#     t0 = time.time()
#     r = portkey.with_options(
#         metadata={
#             "_user": user,
#             'session_id': session,
#             'feature': feature,
#             'environment': 'notebook'
#         }
#     ).chat.completions.create(
#         model=GROQ_MODEL,
#         messages=[{'role':'user','content':q}]
#     )

#     ms = (time.time() - t0) * 1000
#     print(f"\n 🧑‍🦱 {user:8s} | 🔧 {feature:18s} | {ms:.0f}ms")
#     print(f" Q: {q}")
#     print(f" A: {r.choices[0].message.content[:120]}...")

# print("\n✅ Dashboard now shows: cost per user, cells per feature, session grouping")


=*62
EXP 2 - Metadata & Observability

=*62


# EXPERIMENT - 3 Automatic Retries

In [7]:
retrie_config = {
    "retry": {
        "attempts": 3,
        "on_status_codes": [429,500,502,503,504]
    }
}

portkey_retry  = Portkey(api_key=PORTKEY_API_KEY, config=retrie_config)

section("EXP 3 - Automatic Retries")
print("Config: 3 retry attempts on [429,500,502,503,504]")
print("Retries fire automatically on failure - transparent to your code")


# try:
#     to = time.time()
#     r = portkey_retry.chat.completions.create(
#         model=GROQ_MODEL,
#         messages=[{'role':'user', "content":"What is a Kubernets DaemonSet?"}]
#     )
#     ms = (time.time() - t0) * 1000
#     print(f"✅ Succeeded in {ms:.0f}ms")
#     print(f" {r.choices[0].message.content[:300]}")
#     print("\n Retry sequence if Groq had Failed:")
#     print(" Attempt 1 -> 429 wait 1s -> Attempt 2 -> 429 -> wait 2s -> Attemt 3")
#     print(" Your code only sees the final success or the last failure")

# except Exception as e:
#     print(f"❌ All attempts failed : {e}")


=*62
EXP 3 - Automatic Retries

=*62
Config: 3 retry attempts on [429,500,502,503,504]
Retries fire automatically on failure - transparent to your code


# EXPERIMENT 5 -- Fallbacks

Goal: If the primary Groq model fails, automatically switch to the smaller Groq fallback. Users never see the failure.

In [8]:
fallback_config = {
    "strategy": {'mode': "fallback"},
    'targets':[
        {'override_params': {'model': GROQ_MODEL}}, # primary
        {"override_params": {"model": GROQ_MODEL_SMALL}}
    ]
}

portkey_fallback = Portkey(api_key=PORTKEY_API_KEY, config=fallback_config)

section("EXP 5 - Fallback Routing")
print(f'Strategy: {GROQ_MODEL} -> {GROQ_MODEL_SMALL} on failrue \n')

fallback_question = [
    "what is Intel QuickAssist Technology?",
    "Explain Kubernetes persistent value claims."
]

# for  q in fallback_question:
#     try:

#         t0 = time.time()
#         r = portkey_fallback.chat.completions.create(
#             messages=[{'role':"user", 'content':q}]
#         )

#         ms = (time.time() - t0) * 1000
#         show(q, r.choices[0].message.content, ms, label='fallback ready')
#     except Exception as e:
#         print(f"❌ {e}")
#         print(" -> Check that both GROQ_SLUG and GROQ_SLUG_2 Are set correctly in c04")



=*62
EXP 5 - Fallback Routing

=*62
Strategy: @rag/llama-3.3-70b-versatile -> @brag/llama-3.1-8b-instant on failrue 



# EXPERIMENT 6 -- LOAD BALANCING

Goal: Split traffic between providers by weight. Use for gradual migration, A/B testing, or cost control.

New concept: strategy.mode = "loadbalance" with weight per target. Portkey normalises weights to percentrages.

In [9]:
load_balance_config = {
    "strategy":{"mode":"loadbalance"},
    "targets":[
        {"override_params":{"model":GROQ_MODEL}, "weight":0.7}, # 70%
        {"override_params":{"model":GROQ_MODEL_SMALL}, "weight":0.3} # 30%
    ]
}

portkey_lb = Portkey(api_key=PORTKEY_API_KEY, config=load_balance_config)

section("EXP 6 - Load Balancing (70% large / 30% small)")

lb_question = [
    "What is a Kubernetes Ingress resource?",
    "How does OSPF differ from BGP?",
    "What is Intel FPGA acceleration?",
    "Explain Kubernetes HPA.",
    "What is a VLAN trunk?",
    "How does Kubernetes etcd work?"
]

# for i ,q in enumerate(lb_question,1):
#     try:
#         t0 = time.time()
#         r = portkey_lb.chat.completions.create(
#             messages=[{'role':"user", 'content': q}]
#         )

#         ms = (time.time() - t0) * 1000
#         print(f"Req {i} [{ms:.0f}ms]:{q}")
#         print(f"   {r.choices[0].message.content[:120]}....")
#     except Exception as e:
#         print(f"Req {i}: ERROR - {e}")

# print("\n ✅ Check Portkey Logs to see which provider served each request")


=*62
EXP 6 - Load Balancing (70% large / 30% small)

=*62


## EXPERIMENT 7 - Request Caching

Goal: Cache responses to idnetical questions. Second call is instant and const nothing.

New concept: chache.mode = "simple" does exact-mathch caching. identical request -> served from cache, no LLM call.

In [10]:
cache_config = {'cache':{"mode":'simple'}}


portkey_cached = Portkey(api_key=PORTKEY_API_KEY, config=cache_config)

section("EXP 7 - Simple Caching")


# Use a short, precise question with temperature=0 to maximise cache-key stability

q= "Define Kubernetes ConfigMap in one sentence."

call_params = dict(
    model=GROQ_MODEL,
    messages = [{'role':'user', 'content': q}],
    tempreature=0,
    max_tokens=120
)


# print("--- CALL 1: Cache MISS - PortKey forwards to Groq ---")
# t0 = time.time()
# r1 = portkey_cached.chat.completions.create(**call_params)
# t1 = (time.time() - t0) * 1000
# ans1 = r1.choices[0].message.content.strip()
# print(f"Answer : {ans1[:200]}")
# print(f'Latency : {t1:0f}ms | Cost: normal token price')


# print("--- CALL 2: Same Request - should be a cache Hit ---")
# t0 = time.time()
# r2 = portkey_cached.chat.completions.create(**call_params)
# t2 = (time.time() - t0) * 1000
# ans2 = r2.choices[0].message.content.strip()
# print(f"Answer : {ans2[:200]}")
# print(f'Latency : {t2:0f}ms | Cost: normal token price')



=*62
EXP 7 - Simple Caching

=*62


In [11]:
cache_config = {'cache':{"mode":'semantic'}}


portkey_cached = Portkey(api_key=PORTKEY_API_KEY, config=cache_config)

section("EXP 7 - Semantic Caching")


q= "Define Kubernetes autoscaling"

call_params = dict(
    model=GROQ_MODEL,
    messages = [{'role':'user', 'content': q}],
    tempreature=0,
    max_tokens=120
)

# print("--- CALL 1: Cache MISS - PortKey forwards to Groq ---")
# t0 = time.time()
# r1 = portkey_cached.chat.completions.create(**call_params)
# t1 = (time.time() - t0) * 1000
# ans1 = r1.choices[0].message.content.strip()
# print(f"Answer : {ans1[:200]}")
# print(f'Latency : {t1:0f}ms | Cost: normal token price')



=*62
EXP 7 - Semantic Caching

=*62


In [12]:
# q= "What is Kubernetes autoscaling"

# print("--- CALL 2: Same Request - should be a cache Hit ---")
# t0 = time.time()
# r2 = portkey_cached.chat.completions.create(**call_params)
# t2 = (time.time() - t0) * 1000
# ans2 = r2.choices[0].message.content.strip()
# print(f"Answer : {ans2[:200]}")
# print(f'Latency : {t2:0f}ms | Cost: normal token price')

## EXPERIMENT 8 -- Saved Config from Dashboard

Goal: Store your routing strategy in the Portkey dashboard once. Reference it by config ID everywhere. Changes strategy centrally without redeploying.

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


portkey_llm = ChatOpenAI(
    api_key=PORTKEY_API_KEY,
    base_url=PORTKEY_GATEWAY_URL,
    model=GROQ_MODEL,
    temperature=0,
    default_headers=createHeaders(
        api_key=PORTKEY_API_KEY,
        metadata={
            "_user": "rag-pipeline",
            "environment": "notebook",
            "feature": "langchain-integration"
        }
    )
)

section("EXP 9 — LangChain Drop-in")

# Test 1: Direct invoke (like app/agents/nodes/planner.py)
print("--- Test 1: Direct invoke (like planner node) ---")
t0 = time.time()
r = portkey_llm.invoke([
    SystemMessage(content="You are an Enterprise IT Assistant."),
    HumanMessage(content="What is the difference between a Deployment and a StatefulSet?")
])
ms = (time.time() - t0) * 1000
print(f"✅ {ms:.0f}ms")
print(r.content[:250])

# Test 2: LCEL chain (like app/agents/nodes/responder.py)
print("\n--- Test 2: LCEL chain (like responder node) ---")
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Enterprise IT expert. Be concise."),
    ("human",  "{question}")
])
chain = prompt | portkey_llm | StrOutputParser()

t0 = time.time()
answer = chain.invoke({"question": "Explain Kubernetes pod affinity rules."})
ms = (time.time() - t0) * 1000
print(f"✅ {ms:.0f}ms")
print(answer[:250])

print("\n→ This is a drop-in for ChatGroq in app/agents/nodes/*.py")
print("→ Every RAG pipeline call is now logged in Portkey with metadata")


=*62
EXP 9 — LangChain Drop-in

=*62
--- Test 1: Direct invoke (like planner node) ---
✅ 3290ms
In Kubernetes, both Deployments and StatefulSets are used to manage and scale pods, but they serve different purposes and have distinct characteristics.

**Deployments:**

A Deployment is a Kubernetes object that manages a set of replica pods, ensuri

--- Test 2: LCEL chain (like responder node) ---
✅ 2512ms
Kubernetes pod affinity rules define how pods are scheduled and placed on nodes based on their affinity or anti-affinity with other pods. There are two types:

1. **Pod Affinity**: Pods are scheduled on the same node as other pods that match the spec

→ This is a drop-in for ChatGroq in app/agents/nodes/*.py
→ Every RAG pipeline call is now logged in Portkey with metadata
